In [3]:
from reactit.reactit import ReactionGenerator
import itertools as it 
import pandas as pd 
import pubchempy as pcp
from collections import OrderedDict

1. Find possible reaction intermediates given an overall reaction
2. break a reaction down into smaller possible reactions i.e. 

> 2H2 + O2 = 2H2O : 
>
> a) H2 + H2 = H4 
>
> b) H2 + O2 = H2O2 
>
> c) H2O + H2O = ?
>
> (_should be able to go both ways_)

3. find if such combinations exist in a pubchem database
4. calculate the energies of the intermediates and find the lowest energy pathway that can be used for NEB calculations

In [4]:
from reactit.reactit import ReactionGenerator

rg = ReactionGenerator(["H2O","H2","O2"])
rg.iterate()
reaction_dict = rg.as_dict()
reaction_dict

  0%|          | 0/1 [00:00<?, ?it/s]

{0: {'reaction_string': '2 H2O = 2 H2 + 1 O2',
  'reactants': {'H2O': 2},
  'products': {'H2': 2, 'O2': 1}}}

find all possible combinations of reactants with reactants and products with products

In [8]:
def get_combinations(reactions_dict):
    reactants = reactions_dict["reactants"]
    products = reactions_dict["products"]
    list_of_species_reactants = []
    for k, v in reactants.items():
        for i in range(v):
            list_of_species_reactants.append(k)

    combinations_reactants = list(
        dict.fromkeys(
            [tuple(sorted(x))
             for x in list(it.combinations(list_of_species_reactants, 2))]
        )
    )
    list_of_species_products = []
    for k, v in products.items():
        for i in range(v):
            list_of_species_products.append(k)

    combinations_products = list(
        dict.fromkeys(
            [tuple(sorted(x))
             for x in list(it.combinations(list_of_species_products, 2))]
        )
    )

    return combinations_reactants, combinations_products


combinations_reactants, combinations_products = get_combinations(reaction_dict[0])
combinations_reactants,combinations_products

([('H2O', 'H2O')], [('H2', 'H2'), ('H2', 'O2')])

combine the strings into a combination to search the database (takes Hill ordering of elements)

In [10]:
def get_molecule_sums(combinations: list, return_string: bool = True):
    strings = []
    pps = []
    order_map = ["C", "H", "N", "O", "S"]
    for combination in combinations:
        pmol1 = ReactionGenerator.parse_molecule(combination[0])
        pmol2 = ReactionGenerator.parse_molecule(combination[1])

        pp = {key: pmol1.get(key, 0) + pmol2.get(key, 0)
              for key in set(pmol1) | set(pmol2)}
        pp = OrderedDict(
            # hill order
            sorted(pp.items(), key=lambda i: order_map.index(i[0])))
        pps.append(pp)
        molecule_string = ""
        for k, v in pp.items():
            molecule_string += f"{k}{v}"

        strings.append(molecule_string)
    if return_string:
        return (strings)
    else:
        return (pps)


rsums, psums = [
    get_molecule_sums(combinations_reactants, return_string=True), get_molecule_sums(
        combinations_products, return_string=True)
]
psums, rsums

(['H4', 'H2O2'], ['H4O2'])

check whether the formula exists in pubchem

In [11]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

def is_sensible_intermediate(smiles: str) -> bool:
    """
    Return True only if the SMILES represents a single connected,
    neutral, covalently-bonded molecule (not an ion, salt, or fragment mixture).
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False

    # 1. No disconnected fragments (no '.' in parsed mol)
    frags = Chem.GetMolFrags(mol)
    if len(frags) > 1:
        return False          # e.g. [HH].[HH] — two separate molecules

    # 2. No formal charges (filters out ions like [H+], [H-])
    total_charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    if total_charge != 0:
        return False

    # 3. No radicals (optional — remove if you WANT radical intermediates)
    total_radicals = sum(atom.GetNumRadicalElectrons() for atom in mol.GetAtoms())
    if total_radicals > 0:
        return False

    return True

def formula_exists_in_pubchem(formula: str):
    try:
        results = pcp.get_compounds(formula, 'formula')
        smiles = [x.connectivity_smiles for x in results if x.connectivity_smiles]
        sensible = list(dict.fromkeys([s for s in smiles if is_sensible_intermediate(s)]))
        return len(sensible) > 0, sensible
    except Exception as e:
        print("PubChem query failed:", e)
        return False, []

In [14]:
psums[0]

'H4'

In [15]:
import warnings 
warnings.simplefilter("ignore")

In [16]:
product_intermediates = {}
for possible in psums:
    product_intermediates[possible] = formula_exists_in_pubchem(possible)

reactant_intermediates = {}
for possible in rsums:
    reactant_intermediates[possible] = formula_exists_in_pubchem(possible)

[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not removing hydrogen atom without neighbors
[11:32:41] WARNING: not r

In [18]:
reactant_intermediates,product_intermediates

({'H4O2': (False, [])},
 {'H4': (False, []), 'H2O2': (True, ['OO', '[OH2+][O-]'])})

Here you can see that effectively you only have: 

1. H2 + O2 = H2O2 
2. H2 + O2 = H2O+ + O-

In [19]:
reaction_dict

{0: {'reaction_string': '2 H2O = 2 H2 + 1 O2',
  'reactants': {'H2O': 2},
  'products': {'H2': 2, 'O2': 1}}}

In [39]:
rs = pd.Series(reaction_dict[0]["reactants"])
ps = -pd.Series(reaction_dict[0]["products"])
from_products = True
if from_products:
    rs = -rs
    ps = -ps 

df = pd.concat([pd.Series(rs),pd.Series(ps)])
df

H2O   -2
H2     2
O2     1
dtype: int64

In [ ]:
def round_up_species(intermediates, total_species,from_products=False):
    return None